## Open In Colab

[![Run in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/waymax-simulation-experiments/blob/main/experiments/risk-uq-suite/notebooks/decision_audit/oracle_bottleneck_colab.ipynb)


# Oracle Bottleneck Analysis (Colab)

## Objective
Diagnose where performance bottlenecks come from when using the same candidate sets:

- `current` controller (current thresholded rule with predicted risk)
- `oracle-risk` controller (uses true risk label for safety-first selection)
- `oracle-best` controller (best achievable progress among truly safe candidates)


## Hypotheses Tested

1. If `oracle-risk` beats `current` on failure with same candidates, bottleneck is signal/calibration quality.
2. If `oracle-best` adds progress over `oracle-risk` at similar failure, bottleneck is decision rule design.
3. If even oracle controllers remain poor, bottleneck is candidate set quality.


## Step 1 - Deterministic Bootstrap Constants


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/achyutmorang/waymax-simulation-experiments.git'
REPO_DIR = '/content/waymax-simulation-experiments'
REPO_BRANCH = 'main'
EXPERIMENT_SLUG = 'risk-uq-suite'
EXPERIMENT_CONFIG_PATH = f'{REPO_DIR}/configs/experiments/{EXPERIMENT_SLUG}.json'
REQUIRED_DRIVE_FOLDER = '/content/drive/MyDrive/waymax_experiments'

NOTEBOOK_NAME = 'oracle_bottleneck_colab'

runtime_cfg_overrides = {
    'verify_drive_access_every_run': False,
    'force_reinstall': False,
    'auto_restart_after_setup': True,
    'strict_lockfile_check': True,
    'setup_cache_enabled': True,
    'revalidate_core_imports_on_cache_hit': True,
    'setup_cache_path': '/content/.closedloop_setup_cache.json',
    'force_module_hot_reload': True,
}

content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))


## Step 2 - Storage Setup


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    print('[storage] google.colab drive mount unavailable:', exc)

drive_root = Path('/content/drive/MyDrive')
if not drive_root.exists():
    raise RuntimeError('Drive root not found at /content/drive/MyDrive. Ensure Google Drive is mounted.')

required_root = Path(REQUIRED_DRIVE_FOLDER)
required_root.mkdir(parents=True, exist_ok=True)
probe = required_root / '.risk_uq_storage_probe'
probe.write_text('ok')
probe.unlink(missing_ok=True)
print('[storage] ready:', required_root)


## Step 3 - Repo Sync + Runtime Bootstrap


In [ ]:
if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

for p in (REPO_DIR, f'{REPO_DIR}/src'):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.platform import ColabRuntimeConfig, bootstrap_colab_runtime_with_config

runtime_cfg = ColabRuntimeConfig(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    repo_branch=REPO_BRANCH,
    required_drive_folder=REQUIRED_DRIVE_FOLDER,
    **runtime_cfg_overrides,
)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)

globals()['_RISK_UQ_REPO_REV'] = str(bootstrap.repo_sync.repo_rev)
setup_result = bootstrap.setup.result

print('Working directory:', os.getcwd())
print('Repo commit:', bootstrap.repo_sync.repo_rev)
print('Drive mounted:', bootstrap.drive_status.mounted)
print('[setup] result:', setup_result)
if bool(setup_result.get('restart_required', False)) and (not bool(runtime_cfg.auto_restart_after_setup)):
    print('[setup] restart_required=True. Re-run this notebook after runtime restart.')


## Step 4 - Configuration + Run Context


In [ ]:
import numpy as np
import pandas as pd

from src.workflows import (
    initialize_risk_uq_run_context,
    load_experiment_config,
    report_risk_uq_run_context,
)

EXPERIMENT_CFG = load_experiment_config(
    slug=EXPERIMENT_SLUG,
    repo_root=REPO_DIR,
    default_on_missing=False,
)
run_cfg = dict(EXPERIMENT_CFG.get('run', {}))

RUN_NAME = str(run_cfg.get('run_name', '')).strip()
RUN_PREFIX = str(run_cfg.get('run_prefix', 'risk_uq')).strip() or 'risk_uq'
PERSIST_ROOT = str(run_cfg.get('persist_root', '/content/drive/MyDrive/waymax_experiments/risk_uq_runs/v1')).strip()
N_SHARDS = int(max(1, int(run_cfg.get('n_shards', 1))))
SHARD_ID = run_cfg.get('shard_id', 'auto')
RESUME_FROM_EXISTING = bool(run_cfg.get('resume_from_existing', True))
RUN_ENABLED = bool(run_cfg.get('run_enabled', True))

RUN_TAG_PREFIX = str(run_cfg.get('run_tag_prefix', RUN_PREFIX)).strip() or RUN_PREFIX
RUN_MODE_CFG = str(run_cfg.get('run_mode', 'auto')).strip().lower() or 'auto'
RUN_MODE = RUN_MODE_CFG if RESUME_FROM_EXISTING else 'fresh'
RUN_TAG = RUN_NAME

run_ctx = initialize_risk_uq_run_context(
    run_tag=RUN_TAG,
    run_tag_prefix=RUN_TAG_PREFIX,
    persist_root=PERSIST_ROOT,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    resume_mode=RUN_MODE,
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    planner_backend='latentdriver',
)

cfg = run_ctx.cfg
search_cfg = {}
RUN_TAG = str(run_ctx.run_tag)
RUN_NAME = str(RUN_NAME or RUN_TAG)
SHARD_ID = int(run_ctx.shard_id)

print('EXPERIMENT_CONFIG_PATH =', EXPERIMENT_CONFIG_PATH)
print('run_prefix (flat artifacts) =', cfg.run_prefix)
print('RUN_PREFIX/RUN_NAME (contract dir) =', RUN_PREFIX, RUN_NAME)
print('RUN_ENABLED =', RUN_ENABLED, ' RESUME_FROM_EXISTING =', RESUME_FROM_EXISTING)
report_risk_uq_run_context(run_ctx, display_fn=display)

FOCUS_LABEL = str(run_cfg.get('focus_label', 'failure_proxy_h15'))
RISK_BUDGET_TAU = float(run_cfg.get('risk_budget_tau', 0.20))
TAU_SWEEP_VALUES = [round(float(x), 3) for x in list(np.linspace(0.05, 0.80, 16))]
BOOTSTRAP_SAMPLES = int(run_cfg.get('decision_bootstrap_samples', 300))
BOOTSTRAP_SEED = int(run_cfg.get('decision_bootstrap_seed', 17))

CURRENT_RISK_COL_CANDIDATES = [
    str(run_cfg.get('current_risk_col', '')).strip(),
    'risk_combo_platt',
    'planner_risk_combo_platt',
    'risk_combo_raw',
    'planner_risk_combo_proxy',
    'planner_risk_top1_platt',
    'planner_risk_top1_proxy',
]
CURRENT_RISK_COL_CANDIDATES = [c for c in CURRENT_RISK_COL_CANDIDATES if c]


## Step 5 - Load Candidate-Level Data (same candidate sets for all controllers)


In [ ]:
from pathlib import Path
from src.workflows import (
    discover_probe_run_prefixes,
    has_existing_miscalibration_probe_artifacts,
    load_existing_miscalibration_probe_bundle,
)

ANALYSIS_RUN_PREFIX_OVERRIDE = str(run_cfg.get('analysis_run_prefix', '')).strip()
analysis_run_prefix = ANALYSIS_RUN_PREFIX_OVERRIDE or str(cfg.run_prefix)

discovered_runs_df = discover_probe_run_prefixes(PERSIST_ROOT, limit=50)
if not discovered_runs_df.empty:
    display(discovered_runs_df.head(20))

# Strict mode: do NOT fall back to any previous run prefixes.
# analysis_run_prefix stays bound to current run context (or explicit override).

print('analysis_run_prefix =', analysis_run_prefix)

# Prefer cross-signal predictions (contains extra calibrated columns). Fallback to probe predictions.
cross_signal_pred_path = Path(f'{analysis_run_prefix}_cross_signal_decision_predictions.parquet')
if cross_signal_pred_path.exists():
    data_df = pd.read_parquet(cross_signal_pred_path)
    data_source = 'cross_signal_decision_predictions'
else:
    bundle = load_existing_miscalibration_probe_bundle(analysis_run_prefix)
    data_df = bundle.predictions_df.copy()
    data_source = 'miscalibration_probe_predictions'

if data_df.empty:
    raise RuntimeError('Loaded candidate dataset is empty.')

if 'shift_suite' not in data_df.columns:
    data_df['shift_suite'] = 'nominal_clean'
else:
    data_df['shift_suite'] = data_df['shift_suite'].fillna('nominal_clean').astype(str)

if 'scenario_id' not in data_df.columns:
    data_df['scenario_id'] = np.arange(len(data_df)).astype(str)
data_df['scenario_id'] = data_df['scenario_id'].astype(str)

if 'step_idx' not in data_df.columns:
    data_df['step_idx'] = 0
data_df['step_idx'] = pd.to_numeric(data_df['step_idx'], errors='coerce').fillna(0).astype(int)

# Scenario-level split assumptions: evaluate test + holdout when available.
if 'eval_split' in data_df.columns:
    eval_mask = data_df['eval_split'].astype(str).isin(['test', 'high_interaction_holdout'])
    eval_df = data_df.loc[eval_mask].copy()
    if eval_df.empty:
        eval_df = data_df.copy()
        split_source = 'eval_split existed but test/holdout empty; fallback to all rows'
    else:
        split_source = 'eval_split test + high_interaction_holdout'
else:
    eval_df = data_df.copy()
    split_source = 'eval_split missing; fallback to all rows'

if FOCUS_LABEL not in eval_df.columns:
    raise ValueError(f'Focus label {FOCUS_LABEL!r} is missing from candidate data.')

progress_col = 'progress_h6' if 'progress_h6' in eval_df.columns else None
if progress_col is None:
    raise ValueError('Required progress column progress_h6 is missing.')

risk_col = None
for c in CURRENT_RISK_COL_CANDIDATES:
    if c in eval_df.columns:
        risk_col = c
        break

if risk_col is None:
    # Fallback derive combo proxy from dist columns if possible.
    if ('dist_top1_weight' in eval_df.columns) and ('dist_entropy' in eval_df.columns) and ('dist_num_components' in eval_df.columns):
        top1 = np.clip(pd.to_numeric(eval_df['dist_top1_weight'], errors='coerce').fillna(0.5).to_numpy(dtype=float), 1e-6, 1 - 1e-6)
        entropy = pd.to_numeric(eval_df['dist_entropy'], errors='coerce').fillna(0.0).to_numpy(dtype=float)
        n_comp = np.maximum(2.0, pd.to_numeric(eval_df['dist_num_components'], errors='coerce').fillna(2.0).to_numpy(dtype=float))
        entropy_max = np.log(n_comp)
        entropy_max = np.where(np.isfinite(entropy_max) & (entropy_max > 1e-8), entropy_max, 1.0)
        conf_entropy = 1.0 - np.clip(entropy / entropy_max, 0.0, 1.0)
        conf_combo = np.clip(0.7 * top1 + 0.3 * conf_entropy, 0.0, 1.0)
        eval_df['planner_risk_combo_proxy'] = 1.0 - conf_combo
        risk_col = 'planner_risk_combo_proxy'
    else:
        raise RuntimeError('Could not resolve any current risk column for current controller.')

eval_df['risk_current'] = np.clip(pd.to_numeric(eval_df[risk_col], errors='coerce').fillna(0.5).to_numpy(dtype=float), 1e-6, 1 - 1e-6)
eval_df['y_true'] = (pd.to_numeric(eval_df[FOCUS_LABEL], errors='coerce').fillna(0.0).to_numpy(dtype=float) > 0.5).astype(int)
eval_df['step_key'] = eval_df['scenario_id'] + '::' + eval_df['step_idx'].astype(str)

# comfort proxy for tie-breaks.
comfort = np.zeros((len(eval_df),), dtype=float)
if 'max_abs_acc_h6' in eval_df.columns:
    comfort += np.abs(pd.to_numeric(eval_df['max_abs_acc_h6'], errors='coerce').fillna(0.0).to_numpy(dtype=float))
if 'max_abs_jerk_h6' in eval_df.columns:
    comfort += np.abs(pd.to_numeric(eval_df['max_abs_jerk_h6'], errors='coerce').fillna(0.0).to_numpy(dtype=float))
eval_df['comfort_cost'] = comfort

print('data_source =', data_source)
print('split_source =', split_source)
print('rows =', len(eval_df), 'scenarios =', eval_df['scenario_id'].nunique(), 'steps =', eval_df['step_key'].nunique())
print('risk_col(current) =', risk_col)
display(eval_df[['scenario_id', 'step_idx', 'shift_suite', 'risk_current', 'y_true', progress_col, 'comfort_cost']].head(10))


## Step 6 - Controller Definitions and Evaluation Utilities


In [ ]:
import numpy as np
import pandas as pd

def _select_current(step_df, tau):
    # Current rule: budget-gated max-progress; fallback to min predicted risk.
    accepted = step_df['risk_current'].to_numpy(dtype=float) <= float(tau)
    if np.any(accepted):
        sub = step_df.loc[accepted].copy()
        idx = sub['progress_h6'].astype(float).idxmax()
        fallback = 0
    else:
        idx = step_df['risk_current'].astype(float).idxmin()
        fallback = 1
    row = step_df.loc[int(idx)]
    return {
        'selected_idx': int(idx),
        'selected_risk': float(row['risk_current']),
        'accepted': int(float(row['risk_current']) <= float(tau)),
        'fallback': int(fallback),
        'failure': int(row['y_true']),
        'progress': float(row['progress_h6']),
        'comfort': float(row['comfort_cost']),
        'selected_safe': int(int(row['y_true']) == 0),
    }

def _select_oracle_risk(step_df, tau):
    # Oracle-risk: true safety first; conservative tie-break (lowest comfort cost).
    safe = step_df['y_true'].to_numpy(dtype=int) == 0
    if np.any(safe):
        sub = step_df.loc[safe].copy()
        idx = sub['comfort_cost'].astype(float).idxmin()
        fallback = 0
    else:
        idx = step_df['comfort_cost'].astype(float).idxmin()
        fallback = 1
    row = step_df.loc[int(idx)]
    oracle_risk = float(row['y_true'])
    return {
        'selected_idx': int(idx),
        'selected_risk': float(oracle_risk),
        'accepted': int(oracle_risk <= float(tau)),
        'fallback': int(fallback),
        'failure': int(row['y_true']),
        'progress': float(row['progress_h6']),
        'comfort': float(row['comfort_cost']),
        'selected_safe': int(int(row['y_true']) == 0),
    }

def _select_oracle_best(step_df, tau):
    # Oracle-best: among truly safe candidates, maximize progress; otherwise max progress overall.
    safe = step_df['y_true'].to_numpy(dtype=int) == 0
    if np.any(safe):
        sub = step_df.loc[safe].copy()
        idx = sub['progress_h6'].astype(float).idxmax()
        fallback = 0
    else:
        idx = step_df['progress_h6'].astype(float).idxmax()
        fallback = 1
    row = step_df.loc[int(idx)]
    oracle_risk = float(row['y_true'])
    return {
        'selected_idx': int(idx),
        'selected_risk': float(oracle_risk),
        'accepted': int(oracle_risk <= float(tau)),
        'fallback': int(fallback),
        'failure': int(row['y_true']),
        'progress': float(row['progress_h6']),
        'comfort': float(row['comfort_cost']),
        'selected_safe': int(int(row['y_true']) == 0),
    }

CONTROLLERS = {
    'current': _select_current,
    'oracle-risk': _select_oracle_risk,
    'oracle-best': _select_oracle_best,
}

def _evaluate_controller(df, controller_name, tau):
    selector = CONTROLLERS[controller_name]
    rows = []
    for (sid, step_idx, shift_suite), step_df in df.groupby(['scenario_id', 'step_idx', 'shift_suite'], sort=False):
        out = selector(step_df, tau=float(tau))
        safe_exists = int(np.any(step_df['y_true'].to_numpy(dtype=int) == 0))
        min_risk_current = float(np.min(step_df['risk_current'].to_numpy(dtype=float)))
        feasible_current = int(min_risk_current <= float(tau))
        rows.append({
            'scenario_id': str(sid),
            'step_idx': int(step_idx),
            'shift_suite': str(shift_suite),
            'controller': str(controller_name),
            'tau': float(tau),
            'safe_candidate_exists': int(safe_exists),
            'feasible_current': int(feasible_current),
            **out,
        })
    step_df = pd.DataFrame(rows)
    if step_df.empty:
        return step_df, {}

    accepted = step_df['accepted'].to_numpy(dtype=int) == 1
    rejected = ~accepted
    y = step_df['failure'].to_numpy(dtype=float)

    metrics = {
        'n_steps': int(len(step_df)),
        'n_scenarios': int(step_df['scenario_id'].nunique()),
        'failure_rate': float(np.mean(y)),
        'progress_mean': float(np.mean(step_df['progress'].to_numpy(dtype=float))),
        'accept_rate': float(np.mean(accepted)),
        'false_safe': float(np.mean(y[accepted])) if int(np.sum(accepted)) > 0 else np.nan,
        'safe_reject': float(np.mean((y <= 0.5)[rejected])) if int(np.sum(rejected)) > 0 else np.nan,
        'feasible_set_rate': float(np.mean(step_df['feasible_current'].to_numpy(dtype=float))),
        'fallback_rate': float(np.mean(step_df['fallback'].to_numpy(dtype=float))),
        'safe_candidate_rate': float(np.mean(step_df['safe_candidate_exists'].to_numpy(dtype=float))),
    }
    return step_df, metrics

def _bootstrap_ci(df_steps, controller_name, tau, n_boot=0, seed=17):
    n_boot = int(max(0, int(n_boot)))
    if (n_boot <= 0) or df_steps.empty:
        return {}
    sid = df_steps['scenario_id'].astype(str)
    groups = {k: v.index.to_numpy(dtype=int) for k, v in df_steps.groupby(sid, sort=False)}
    keys = list(groups.keys())
    if len(keys) <= 1:
        return {}
    rng = np.random.default_rng(int(seed))
    vals = []
    for _ in range(n_boot):
        picked = rng.integers(0, len(keys), size=len(keys))
        idx = np.concatenate([groups[keys[int(i)]] for i in picked], axis=0)
        sub = df_steps.iloc[idx]
        accepted = sub['accepted'].to_numpy(dtype=int) == 1
        rejected = ~accepted
        y = sub['failure'].to_numpy(dtype=float)
        vals.append({
            'failure_rate': float(np.mean(y)),
            'progress_mean': float(np.mean(sub['progress'].to_numpy(dtype=float))),
            'accept_rate': float(np.mean(accepted)),
            'false_safe': float(np.mean(y[accepted])) if int(np.sum(accepted)) > 0 else np.nan,
            'safe_reject': float(np.mean((y <= 0.5)[rejected])) if int(np.sum(rejected)) > 0 else np.nan,
            'feasible_set_rate': float(np.mean(sub['feasible_current'].to_numpy(dtype=float))),
            'fallback_rate': float(np.mean(sub['fallback'].to_numpy(dtype=float))),
        })
    out = {}
    for k in vals[0].keys():
        arr = np.asarray([x.get(k, np.nan) for x in vals], dtype=float)
        arr = arr[np.isfinite(arr)]
        if arr.size == 0:
            continue
        out[f'{k}_ci_low'] = float(np.quantile(arr, 0.025))
        out[f'{k}_ci_high'] = float(np.quantile(arr, 0.975))
    return out


## Step 7 - Run Controller Comparisons and Tau Sweep


In [ ]:
import numpy as np
import pandas as pd

all_step_rows = []
all_metric_rows = []

shifts = sorted(eval_df['shift_suite'].astype(str).unique().tolist())

for shift_suite in shifts:
    shift_df = eval_df[eval_df['shift_suite'].astype(str).eq(str(shift_suite))].copy()
    for controller_name in CONTROLLERS.keys():
        for tau in TAU_SWEEP_VALUES:
            step_trace, metrics = _evaluate_controller(shift_df, controller_name=controller_name, tau=float(tau))
            if step_trace.empty:
                continue
            ci = _bootstrap_ci(step_trace, controller_name=controller_name, tau=float(tau), n_boot=BOOTSTRAP_SAMPLES, seed=BOOTSTRAP_SEED)

            gate_reasons = []
            if int(metrics.get('n_steps', 0)) < 40:
                gate_reasons.append('low_steps')
            if int(step_trace['failure'].sum()) < 10:
                gate_reasons.append('low_failures')
            if int(step_trace['accepted'].sum()) < 20:
                gate_reasons.append('low_accept_count')

            all_metric_rows.append({
                'shift_suite': str(shift_suite),
                'domain': 'nominal' if str(shift_suite) == 'nominal_clean' else 'shifted',
                'controller': str(controller_name),
                'tau': float(tau),
                **metrics,
                **ci,
                'status': 'ok' if len(gate_reasons) == 0 else 'inconclusive',
                'inconclusive_reason': ';'.join(gate_reasons),
            })

            step_trace = step_trace.copy()
            step_trace['domain'] = 'nominal' if str(shift_suite) == 'nominal_clean' else 'shifted'
            step_trace['status'] = 'ok' if len(gate_reasons) == 0 else 'inconclusive'
            step_trace['inconclusive_reason'] = ';'.join(gate_reasons)
            all_step_rows.append(step_trace)

ctrl_step_df = pd.concat(all_step_rows, ignore_index=True) if len(all_step_rows) else pd.DataFrame()
ctrl_metrics_df = pd.DataFrame(all_metric_rows)

if ctrl_metrics_df.empty:
    raise RuntimeError('No controller metrics were generated.')

at_budget_df = (
    ctrl_metrics_df.assign(_dist=(ctrl_metrics_df['tau'] - float(RISK_BUDGET_TAU)).abs())
    .sort_values('_dist')
    .groupby(['shift_suite', 'controller'], as_index=False)
    .first()
    .drop(columns=['_dist'])
    .sort_values(['shift_suite', 'controller'])
    .reset_index(drop=True)
)

# Bottleneck diagnosis at tau budget (aggregate on nominal first, fallback all).
focus_diag_df = at_budget_df[at_budget_df['shift_suite'].astype(str).eq('nominal_clean')].copy()
if focus_diag_df.empty:
    focus_diag_df = at_budget_df.copy()

diag = {'signal_or_calibration_bottleneck': np.nan, 'controller_rule_bottleneck': np.nan, 'candidate_quality_bottleneck': np.nan}
diag_evidence = {}

def _row_for(name):
    r = focus_diag_df[focus_diag_df['controller'].astype(str).eq(name)]
    return r.iloc[0] if not r.empty else None

r_cur = _row_for('current')
r_orisk = _row_for('oracle-risk')
r_obest = _row_for('oracle-best')

if (r_cur is not None) and (r_orisk is not None):
    fail_gap = float(r_cur['failure_rate']) - float(r_orisk['failure_rate'])
    diag['signal_or_calibration_bottleneck'] = int(fail_gap > 0.02)
    diag_evidence['signal_fail_gap_current_minus_oracle_risk'] = fail_gap

if (r_orisk is not None) and (r_obest is not None):
    prog_gap = float(r_obest['progress_mean']) - float(r_orisk['progress_mean'])
    fail_gap_ob = float(r_obest['failure_rate']) - float(r_orisk['failure_rate'])
    diag['controller_rule_bottleneck'] = int((prog_gap > 0.02) and (fail_gap_ob <= 0.01))
    diag_evidence['rule_progress_gap_oracle_best_minus_oracle_risk'] = prog_gap
    diag_evidence['rule_failure_gap_oracle_best_minus_oracle_risk'] = fail_gap_ob

if r_obest is not None:
    safe_rate = float(r_obest['safe_candidate_rate']) if np.isfinite(float(r_obest['safe_candidate_rate'])) else np.nan
    fail_rate = float(r_obest['failure_rate'])
    diag['candidate_quality_bottleneck'] = int((np.isfinite(safe_rate) and safe_rate < 0.75) or (fail_rate > 0.15))
    diag_evidence['oracle_best_safe_candidate_rate'] = safe_rate
    diag_evidence['oracle_best_failure_rate'] = fail_rate

diagnosis_df = pd.DataFrame([
    {
        **diag,
        **diag_evidence,
        'note': '1 indicates likely bottleneck present under this offline candidate-set audit.',
    }
])

print('ctrl_metrics_df rows =', len(ctrl_metrics_df))
print('ctrl_step_df rows =', len(ctrl_step_df))
print('inconclusive metric rows =', int((ctrl_metrics_df['status'] == 'inconclusive').sum()))

display(at_budget_df)
display(diagnosis_df)


## Step 8 - Progress vs Failure Tradeoff and Controller Comparison Plots


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

figure_paths = {}

# Plot 1: Progress vs failure tradeoff across tau.
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
for controller, color in [('current', 'tab:blue'), ('oracle-risk', 'tab:red'), ('oracle-best', 'tab:green')]:
    sub = ctrl_metrics_df[ctrl_metrics_df['controller'].astype(str).eq(controller)].copy()
    if sub.empty:
        continue
    sub = (
        sub.groupby('tau', as_index=False)
        .agg(
            failure_rate=('failure_rate', 'mean'),
            progress_mean=('progress_mean', 'mean'),
        )
        .sort_values('tau')
    )
    ax.plot(sub['progress_mean'], sub['failure_rate'], marker='o', linewidth=1.8, label=controller, color=color)

ax.set_title('Progress vs Failure Tradeoff (tau sweep)')
ax.set_xlabel('mean progress')
ax.set_ylabel('failure rate')
ax.grid(alpha=0.25)
ax.legend(loc='best')
fig.tight_layout()
p = f'{analysis_run_prefix}_oracle_bottleneck_tradeoff.png'
fig.savefig(p, dpi=170, bbox_inches='tight')
figure_paths['oracle_bottleneck_tradeoff'] = p
plt.show()

# Plot 2: Budget diagnostics at tau~budget.
metric_cols = ['accept_rate', 'false_safe', 'safe_reject', 'feasible_set_rate', 'fallback_rate']
fig, axes = plt.subplots(1, len(metric_cols), figsize=(4.8 * len(metric_cols), 5))
for ax, metric in zip(axes, metric_cols):
    pivot = at_budget_df.pivot(index='shift_suite', columns='controller', values=metric)
    cols = [c for c in ['current', 'oracle-risk', 'oracle-best'] if c in pivot.columns]
    pivot = pivot.loc[:, cols]
    pivot.plot(kind='bar', ax=ax)
    ax.set_title(metric)
    ax.set_xlabel('shift_suite')
    ax.set_ylabel(metric)
    if metric == 'false_safe':
        ax.axhline(float(RISK_BUDGET_TAU), color='k', linestyle='--', linewidth=1)
    ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
p = f'{analysis_run_prefix}_oracle_bottleneck_budget_metrics.png'
fig.savefig(p, dpi=170, bbox_inches='tight')
figure_paths['oracle_bottleneck_budget_metrics'] = p
plt.show()

# Plot 3: Failure and progress with CIs near budget tau.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for metric, ax in [('failure_rate', axes[0]), ('progress_mean', axes[1])]:
    x = np.arange(len(at_budget_df))
    labels = [f"{r['shift_suite']}\\n{r['controller']}" for _, r in at_budget_df.iterrows()]
    vals = at_budget_df[metric].to_numpy(dtype=float)
    lo = pd.to_numeric(at_budget_df.get(f'{metric}_ci_low', np.nan), errors='coerce').to_numpy(dtype=float)
    hi = pd.to_numeric(at_budget_df.get(f'{metric}_ci_high', np.nan), errors='coerce').to_numpy(dtype=float)
    yerr = np.vstack([
        np.where(np.isfinite(lo), np.maximum(0.0, vals - lo), 0.0),
        np.where(np.isfinite(hi), np.maximum(0.0, hi - vals), 0.0),
    ])
    ax.bar(x, vals, yerr=yerr, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=35, ha='right')
    ax.set_title(metric)
    ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
p = f'{analysis_run_prefix}_oracle_bottleneck_ci_summary.png'
fig.savefig(p, dpi=170, bbox_inches='tight')
figure_paths['oracle_bottleneck_ci_summary'] = p
plt.show()

print('figure_paths =', figure_paths)


## Step 9 - Export Artifacts + Contract Manifest


In [ ]:
from src.workflows import write_contract_storage_mirror, write_notebook_contract_manifest

metrics_path = f'{analysis_run_prefix}_oracle_bottleneck_metrics.csv'
step_path = f'{analysis_run_prefix}_oracle_bottleneck_step_trace.csv'
budget_path = f'{analysis_run_prefix}_oracle_bottleneck_at_budget.csv'
diagnosis_path = f'{analysis_run_prefix}_oracle_bottleneck_diagnosis.csv'

ctrl_metrics_df.to_csv(metrics_path, index=False)
ctrl_step_df.to_csv(step_path, index=False)
at_budget_df.to_csv(budget_path, index=False)
diagnosis_df.to_csv(diagnosis_path, index=False)

artifact_paths = {
    'oracle_bottleneck_metrics': metrics_path,
    'oracle_bottleneck_step_trace': step_path,
    'oracle_bottleneck_at_budget': budget_path,
    'oracle_bottleneck_diagnosis': diagnosis_path,
    **figure_paths,
}

extra = {
    'analysis_run_prefix': str(analysis_run_prefix),
    'focus_label': str(FOCUS_LABEL),
    'risk_budget_tau': float(RISK_BUDGET_TAU),
    'bootstrap_samples': int(BOOTSTRAP_SAMPLES),
    'metrics_rows': int(len(ctrl_metrics_df)),
    'step_rows': int(len(ctrl_step_df)),
    'inconclusive_rows': int((ctrl_metrics_df['status'] == 'inconclusive').sum()) if not ctrl_metrics_df.empty else 0,
    'figure_count': int(len(figure_paths)),
}

manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name=NOTEBOOK_NAME,
    stage='oracle_bottleneck_completed',
    repo_dir=REPO_DIR,
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
    extra_fields=extra,
)

contract_paths = write_contract_storage_mirror(
    persist_root=PERSIST_ROOT,
    run_prefix=RUN_PREFIX,
    run_name=RUN_NAME,
    run_prefix_path=cfg.run_prefix,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    stage='oracle_bottleneck_completed',
    git_commit=str(globals().get('_RISK_UQ_REPO_REV', '')),
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    run_enabled=bool(RUN_ENABLED),
    artifact_paths=artifact_paths,
    metrics_csv_path=metrics_path,
    extra_fields=extra,
)

print('manifest_path =', manifest_path)
print('contract_run_manifest =', contract_paths['contract_run_manifest'])
for k, v in sorted(artifact_paths.items()):
    print('-', k, '->', v)


## Step 10 - Final Summary


In [ ]:
print('=== Final Summary (Oracle Bottleneck Analysis) ===')
print('focus_label =', FOCUS_LABEL, '| tau =', RISK_BUDGET_TAU)

nominal = at_budget_df[at_budget_df['shift_suite'].astype(str).eq('nominal_clean')].copy()
if nominal.empty:
    nominal = at_budget_df.copy()

for controller in ['current', 'oracle-risk', 'oracle-best']:
    sub = nominal[nominal['controller'].astype(str).eq(controller)]
    if sub.empty:
        continue
    r = sub.iloc[0]
    print(f"[{controller}] failure={float(r['failure_rate']):.4f} progress={float(r['progress_mean']):.4f} "
          f"accept={float(r['accept_rate']):.4f} false_safe={float(r['false_safe']) if __import__('numpy').isfinite(float(r['false_safe'])) else float('nan'):.4f} "
          f"safe_reject={float(r['safe_reject']) if __import__('numpy').isfinite(float(r['safe_reject'])) else float('nan'):.4f} "
          f"feasible={float(r['feasible_set_rate']):.4f} fallback={float(r['fallback_rate']):.4f}")

if not diagnosis_df.empty:
    d = diagnosis_df.iloc[0]
    print('\\nLikely bottlenecks (1=yes, 0=no, nan=inconclusive):')
    print(' signal/calibration:', d.get('signal_or_calibration_bottleneck', 'nan'))
    print(' controller rule   :', d.get('controller_rule_bottleneck', 'nan'))
    print(' candidate quality :', d.get('candidate_quality_bottleneck', 'nan'))
    print(' evidence:', {k: d[k] for k in diagnosis_df.columns if k.endswith('gap') or k.endswith('rate')})

print('\\nInterpretation guideline:')
print('- current vs oracle-risk isolates signal/calibration limitations with fixed candidate sets.')
print('- oracle-risk vs oracle-best isolates rule conservatism vs utility exploitation.')
print('- poor oracle-best indicates candidate generation quality is the limiting factor.')
